# Model Selection
## Nine models, three baselines, one honest comparison

Runs on `data/model_train.csv` and `data/model_test.csv` from Part 1. No
network, no credentials.

### What this notebook is careful about

**Effective sample size.** 13,300 training rows, but only **665 distinct
trading days**. The 20 names on a day share one market shock and were demeaned
against each other, so rows within a day are close to the opposite of
independent. Roughly **17 effective observations per feature, not 350**. Every
grid here is deliberately small, every t-statistic is computed across days, and
no impressive number is taken at face value.

**Selection on rank IC, not MAE.** The strategy monetises the *ordering* of
names within a day. A model predicting ~0 everywhere scores the best MAE in the
study and is worth nothing. The `ZeroPredictor` baseline below exists to make
that visible rather than argued.

**The one-standard-error rule.** Nine models scoring within a band narrower
than their own standard errors are not nine distinguishable models. Taking the
maximum of nine correlated noisy estimates is upward-biased even if all are
equally good — and that bias gets paid back the moment the test set is opened.
So the rule is: best mean fold IC, then among everything within one standard
error of it, take the **simplest**.

In [ ]:
import os, sys, json, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Notebooks live in modelling/notebooks/, so the repo root is two levels up.
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if not os.path.isdir(os.path.join(ROOT, "data")):
    ROOT = os.path.abspath(os.getcwd())          # already at the root
sys.path.insert(0, ROOT)

DATA = os.path.join(ROOT, "data")
RESULTS = os.path.join(ROOT, "modelling", "results")
FIGS = os.path.join(RESULTS, "figures")
os.makedirs(FIGS, exist_ok=True)

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

from modelling.src.metrics import (mae, profit_factor, signal_profit_factor,
                                   rank_ic, daily_ic_series, report,
                                   directional_accuracy_by_quantile)
from modelling.src.splitters import DateGroupedTimeSeriesSplit, CombinatorialPurgedCV
from modelling.src.models import build_roster, wrap, RANDOM_STATE
from modelling.src.baselines import (ZeroPredictor, RandomPredictor,
                                     BestSingleFeature, random_ic_null)
from modelling.src.evaluate import build, cv_oof, fold_scores, run_roster, select_one_se

train = pd.read_csv(f"{DATA}/model_train.csv", parse_dates=["date"]).reset_index(drop=True)
test = pd.read_csv(f"{DATA}/model_test.csv", parse_dates=["date"]).reset_index(drop=True)
FEATURES = json.load(open(f"{DATA}/feature_meta.json"))["selected"]

X_tr, y_tr, d_tr = train[FEATURES], train["target"].values, train["date"].values
X_te, y_te, d_te = test[FEATURES], test["target"].values, test["date"].values

# Training target winsorised at +/-5 sd; every score uses the RAW target.
WINSOR_SD = 5.0
_sd = y_tr.std()
y_tr_w = np.clip(y_tr, -WINSOR_SD * _sd, WINSOR_SD * _sd)

print(f"train {X_tr.shape[0]:,} rows x {X_tr.shape[1]} features | "
      f"{train['date'].nunique():,} days | {train['ticker'].nunique()} names")
print(f"test  {X_te.shape[0]:,} rows | {test['date'].nunique():,} days")
print(f"\ntarget sd {y_tr.std():.5f}  kurtosis {pd.Series(y_tr).kurt():.1f}")
print(f"winsorised at +/-{WINSOR_SD} sd -> "
      f"{int((np.abs(y_tr) > WINSOR_SD*_sd).sum())} rows clipped "
      f"({100*(np.abs(y_tr) > WINSOR_SD*_sd).mean():.2f}%), "
      f"kurtosis -> {pd.Series(y_tr_w).kurt():.1f}")
print(f"features carrying NaN: {int(X_tr.isna().any().sum())} of {len(FEATURES)}")

## 1. What does chance look like?

Before any model, fix the zero point. Predictions are shuffled **within each
day**, which preserves the exact panel shape — same days, same cross-section
size, same target distribution — and destroys only the ordering. That makes it
the right null for a cross-sectional ranking score.

In [ ]:
draws, null_mean, null_sd, (null_lo, null_hi) = random_ic_null(
    y_tr, d_tr, n_draws=500, seed=RANDOM_STATE)

print(f"Empirical null over 500 draws ({train['date'].nunique()} days x 20 names):")
print(f"  mean rank IC      {null_mean:+.5f}")
print(f"  standard deviation {null_sd:.5f}")
print(f"  95% interval      [{null_lo:+.4f}, {null_hi:+.4f}]")
print(f"\nAnything inside that interval is indistinguishable from chance.")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(draws, bins=40, color="#7A8C8B", edgecolor="white")
ax.axvline(0, color="black", lw=1)
ax.axvspan(null_lo, null_hi, color="#0D5B63", alpha=.12, label="95% of chance outcomes")
ax.set_xlabel("rank IC under random predictions"); ax.set_ylabel("draws")
ax.set_title("The zero point: what a rank IC of pure chance looks like on this panel")
ax.legend(); plt.tight_layout()
plt.savefig(f"{FIGS}/null_distribution.png", dpi=140, bbox_inches="tight"); plt.show()

## 2. Baselines

Three, each answering a different "compared to what".

`BestSingleFeature` is the one that matters. It picks the strongest single
feature **inside each training fold** (never on the full sample) and uses it
raw, sign-corrected, as the prediction. If nine tuned models cannot beat one
z-scored column, the machine learning added nothing — and the deck should
report that rather than omit the comparison.

In [ ]:
CV = DateGroupedTimeSeriesSplit(n_splits=5, embargo=5)

print("Fold structure (expanding window, 5-day embargo):")
for k, (tr, va) in enumerate(CV.split(X_tr, y_tr, groups=d_tr)):
    td, vd = pd.Series(d_tr[tr]), pd.Series(d_tr[va])
    print(f"  fold {k}: train {len(tr):>6,} / {td.nunique():>3}d "
          f"({td.min():%Y-%m-%d}..{td.max():%Y-%m-%d})  ->  "
          f"val {len(va):>5,} / {vd.nunique():>3}d   gap {(vd.min()-td.max()).days}d")

BASELINES = {
    "ZeroPredictor":     {"factory": lambda: wrap(ZeroPredictor()),
                          "grid": [{}], "family": "baseline", "scaled": False},
    "RandomPredictor":   {"factory": lambda: wrap(RandomPredictor(seed=RANDOM_STATE)),
                          "grid": [{}], "family": "baseline", "scaled": False},
    "BestSingleFeature": {"factory": lambda: wrap(BestSingleFeature(feature_names=FEATURES)),
                          "grid": [{}], "family": "baseline", "scaled": False},
}

print("\nBaselines:")
base_results = run_roster(BASELINES, X_tr, y_tr_w, y_tr, d_tr, CV)

bsf = build(BASELINES["BestSingleFeature"], {})
bsf.fit(X_tr, y_tr_w)
inner = bsf.regressor_
print(f"\nBestSingleFeature picked: {inner.feature_}  "
      f"(train PF {inner.pf_:.4f}, sign {'+' if inner.sign_ > 0 else '-'})")
print(f"  MAE comparison -- ZeroPredictor {base_results['ZeroPredictor']['MAE']:.6f} "
      f"vs BestSingleFeature {base_results['BestSingleFeature']['MAE']:.6f}")
print("  The constant wins on MAE and cannot trade. That is the argument against MAE.")

## 3. The nine models

Four genuinely different inductive biases, so that whichever family wins is a
finding rather than an artefact of only having tried one kind of model.

| Family | Models | Scaling |
|---|---|---|
| Boosted trees | LightGBM, HistGBM | no — native NaN handling |
| Bagged trees | RandomForest, ExtraTrees | no |
| Penalised linear | Ridge, ElasticNet | **yes** |
| Kernel / distance / neural | SVR, kNN, Keras MLP | **yes** |

`StandardScaler` and `SelectKBest` are pipeline steps, refit inside every fold
— the brief is explicit that fitting them outside would leak validation-fold
statistics into training. `f_regression` scores the linear models,
`mutual_info_regression` the non-linear ones.

In [ ]:
roster = build_roster(len(FEATURES))
total_fits = sum(len(s["grid"]) for s in roster.values()) * CV.get_n_splits()
print(f"{'model':14s} {'family':18s} {'configs':>8s} {'scaled':>7s}")
for n, s in roster.items():
    print(f"{n:14s} {s['family']:18s} {len(s['grid']):8d} {str(s['scaled']):>7s}")
print(f"\n{total_fits} fits for the grid search, plus {len(roster)} for out-of-fold prediction.")

t0 = time.time()
results = run_roster(roster, X_tr, y_tr_w, y_tr, d_tr, CV)
print(f"\ncompleted in {(time.time()-t0)/60:.1f} min")

## 4. Comparison and selection

In [ ]:
all_results = {**base_results, **results}
winner, table = select_one_se(all_results)

show = table[["model", "family", "mean_fold_ic", "fold_ic_se", "folds_positive",
              "oof_RankIC", "oof_RankIC_t", "MAE", "ProfitFactor", "DirAcc",
              "Skill", "DegenDays", "within_1se", "selected"]]
print("Ranked by MEAN FOLD rank IC (not pooled):\n")
print(show.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

top = table.iloc[0]
print(f"\nBest mean fold IC : {top['model']} at {top['mean_fold_ic']:+.4f} "
      f"(se {top['fold_ic_se']:.4f})")
print(f"Within 1 s.e.     : {', '.join(table[table['within_1se']]['model'])}")
print(f"SELECTED          : {winner}  (simplest family within one standard error)")

bsf_ic = table.loc[table['model'] == 'BestSingleFeature', 'mean_fold_ic'].iloc[0]
win_ic = table.loc[table['model'] == winner, 'mean_fold_ic'].iloc[0]
print(f"\nvs the single-feature baseline: {win_ic:+.4f} against {bsf_ic:+.4f} "
      f"({win_ic - bsf_ic:+.4f})")
print(f"vs chance (95% null upper bound {null_hi:+.4f}): "
      f"{'clears it' if win_ic > null_hi else 'INSIDE the null -- not distinguishable from chance'}")

table.to_csv(f"{RESULTS}/model_comparison.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 6))
t = table.iloc[::-1]
cols = ["#0D5B63" if s else ("#9BB0AF" if w else "#C9D4D3")
        for s, w in zip(t["selected"], t["within_1se"])]

axes[0].barh(t["model"], t["mean_fold_ic"], color=cols,
             xerr=t["fold_ic_se"], error_kw=dict(ecolor="#55605F", lw=1))
axes[0].axvline(0, color="black", lw=1)
axes[0].axvspan(null_lo, null_hi, color="#8B3F33", alpha=.10)
axes[0].set_title("Mean fold rank IC (+/- 1 s.e.)\nred band = indistinguishable from chance", fontsize=10)

axes[1].barh(t["model"], t["folds_positive"], color=cols)
axes[1].axvline(2.5, color="black", ls="--", lw=1)
axes[1].set_title("Folds with a positive IC (out of 5)\nconsistency, not magnitude", fontsize=10)

axes[2].barh(t["model"], t["MAE"], color=cols)
axes[2].set_title("MAE -- note the ranking is almost the REVERSE\nof rank IC", fontsize=10)
axes[2].set_xlim(t["MAE"].min() * .98, t["MAE"].max() * 1.02)

plt.tight_layout()
plt.savefig(f"{FIGS}/model_comparison.png", dpi=140, bbox_inches="tight"); plt.show()

## 5. Fine-tuning the selected model

A second, finer manual grid around the winner. Kept deliberately small: with
665 effective days, a wide search mostly locates the configuration best suited
to this particular fold split, and that advantage does not survive contact with
the test set.

In [ ]:
spec = roster[winner] if winner in roster else BASELINES[winner]
base_params = all_results[winner]["params"]
print(f"Fine-tuning {winner}; stage-1 best: {base_params}")

FINE = {
    "LightGBM":     [dict(base_params, num_leaves=nl, learning_rate=lr, reg_lambda=rl)
                     for nl in (5, 7, 11) for lr in (0.01, 0.02) for rl in (10.0, 30.0)],
    "HistGBM":      [dict(base_params, max_leaf_nodes=ml, learning_rate=lr, l2_regularization=l2)
                     for ml in (5, 7, 11) for lr in (0.01, 0.02) for l2 in (5.0, 20.0)],
    "RandomForest": [dict(base_params, **{"model__max_depth": d, "model__min_samples_leaf": lf})
                     for d in (3, 4, 6) for lf in (100, 200, 400)],
    "ExtraTrees":   [dict(base_params, **{"model__max_depth": d, "model__min_samples_leaf": lf})
                     for d in (3, 4, 6) for lf in (100, 200, 400)],
    "Ridge":        [dict(base_params, **{"model__alpha": a})
                     for a in (10., 30., 100., 300., 1000., 3000.)],
    "ElasticNet":   [dict(base_params, **{"model__alpha": a, "model__l1_ratio": l1})
                     for a in (3e-4, 1e-3, 3e-3, 1e-2) for l1 in (0.1, 0.5, 0.9)],
    "SVR":          [dict(base_params, **{"model__C": c, "model__epsilon": e})
                     for c in (0.05, 0.1, 0.3) for e in (0.02, 0.05, 0.1)],
    "kNN":          [dict(base_params, **{"model__n_neighbors": nn})
                     for nn in (200, 400, 600, 800, 1200)],
    "KerasMLP":     [dict(base_params, **{"model__dropout": dr, "model__l2": l2})
                     for dr in (0.2, 0.3, 0.4) for l2 in (1e-3, 1e-2)],
}.get(winner, [base_params])

# The stage-1 winner MUST be in the fine grid. Without it, a finer search over a
# neighbourhood that happens to exclude the incumbent can only move sideways or
# down -- and it did: HistGBM's stage-1 best (max_leaf_nodes=15, lr=0.05) fell
# outside the refined ranges, so "tuning" cost 0.0073 of mean fold IC and the
# worse configuration was kept. Tuning must never be able to lose.
FINE = [base_params] + [f for f in FINE if f != base_params]

best_fine, t0 = None, time.time()
for params in FINE:
    ics = fold_scores(spec, params, X_tr, y_tr_w, y_tr, d_tr, CV)
    if best_fine is None or ics.mean() > best_fine["ic"]:
        best_fine = {"params": params, "ic": float(ics.mean()),
                     "se": float(ics.std(ddof=1) / np.sqrt(len(ics))), "ics": ics}
print(f"searched {len(FINE)} configurations (incumbent included) in {time.time()-t0:.0f}s")
stage1_ic = all_results[winner]["mean_fold_ic"]
print(f"  stage-1 incumbent scored {stage1_ic:+.4f}; tuning "
      f"{'improved on it' if best_fine['ic'] > stage1_ic else 'could not beat it (incumbent kept)'}")
print(f"  best mean fold IC {best_fine['ic']:+.4f} (se {best_fine['se']:.4f}), "
      f"positive in {int((best_fine['ics'] > 0).sum())}/{len(best_fine['ics'])} folds")
print(f"  params: {best_fine['params']}")
FINAL_PARAMS = best_fine["params"]
oof_final = cv_oof(spec, FINAL_PARAMS, X_tr, y_tr_w, y_tr, d_tr, CV)

## 6. The held-out test set

Refit on the whole training block with the tuned parameters, predict once.
Nothing below has influenced any choice above.

In [ ]:
final = build(spec, FINAL_PARAMS)
final.fit(X_tr, y_tr_w)
pred_te = final.predict(X_te)

varies = pd.Series(pred_te).groupby(pd.Series(d_te)).nunique().gt(1)
print(f"Test days on which the prediction varies across the 20 names: "
      f"{varies.sum()} of {len(varies)}")
assert varies.mean() > 0.5, "the fitted model gives no cross-sectional ordering"

m = np.isfinite(oof_final)
res_val = report(y_tr[m], oof_final[m], d_tr[m], label=f"{winner} -- VALIDATION (out-of-fold)")
res_te = report(y_te, pred_te, d_te, label=f"{winner} -- TEST (held out)")

In [ ]:
side = pd.DataFrame([
    {"metric": "Mean absolute error",    "validation": res_val["MAE"],          "test": res_te["MAE"]},
    {"metric": "Profit Factor",          "validation": res_val["ProfitFactor"], "test": res_te["ProfitFactor"]},
    {"metric": "Spearman rank IC",       "validation": res_val["RankIC"],       "test": res_te["RankIC"]},
    {"metric": "Rank IC t-stat",         "validation": res_val["RankIC_t"],     "test": res_te["RankIC_t"]},
    {"metric": "Directional accuracy %", "validation": res_val["DirAcc"],       "test": res_te["DirAcc"]},
    {"metric": "Naive baseline %",       "validation": res_val["NaiveDirAcc"],  "test": res_te["NaiveDirAcc"]},
    {"metric": "Skill (pct points)",     "validation": res_val["Skill"],        "test": res_te["Skill"]},
])
print("VALIDATION vs TEST\n")
print(side.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))
side.to_csv(f"{RESULTS}/validation_vs_test.csv", index=False)

print(f"\nRank IC degradation: {res_val['RankIC'] - res_te['RankIC']:+.4f}")

# Every model on the test set -- a DIAGNOSTIC, run after selection was locked
# and fed back into nothing. It answers one question: was the outcome specific
# to the selected model, or did none of them carry?
diag = []
for name, spec_i in roster.items():
    est = build(spec_i, all_results[name]["params"]); est.fit(X_tr, y_tr_w)
    p = est.predict(X_te)
    ic, t, nd, _ = rank_ic(y_te, p, d_te)
    diag.append({"model": name, "val_mean_fold_ic": all_results[name]["mean_fold_ic"],
                 "test_RankIC": ic, "test_t": t,
                 "test_PF": signal_profit_factor(y_te, p)})
diag = pd.DataFrame(diag).sort_values("val_mean_fold_ic", ascending=False)
print("\n\nPOST-HOC DIAGNOSTIC -- every model on the test set (not used for selection):\n")
print(diag.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))
print(f"\nmean test rank IC across all {len(diag)} models: {diag['test_RankIC'].mean():+.4f}")
print(f"models positive out of sample: {int((diag['test_RankIC'] > 0).sum())} of {len(diag)}")
diag.to_csv(f"{RESULTS}/test_diagnostic_all_models.csv", index=False)

In [ ]:
out_te = test[["date", "ticker", "target", "fwd_logret", "close_adj", "stock_spread"]].copy()
out_te["pred"] = pred_te
out_te.to_csv(f"{RESULTS}/pred_test.csv", index=False)

out_tr = train.loc[m, ["date", "ticker", "target", "fwd_logret"]].copy()
out_tr["pred"] = oof_final[m]
out_tr.to_csv(f"{RESULTS}/pred_train_oof.csv", index=False)

# Pickle rather than JSON: several grids carry tuples (e.g. hidden=(64,32))
# and a JSON round-trip would hand notebook 2 a list, which Keras rejects.
import pickle
with open(f"{RESULTS}/params.pkl", "wb") as fh:
    pickle.dump({"winner": winner,
                 "final_params": FINAL_PARAMS,
                 "best_params_by_model": {n: r["params"] for n, r in all_results.items()}}, fh)

json.dump({"winner": winner, "family": all_results[winner]["family"],
           "params": {k: str(v) for k, v in FINAL_PARAMS.items()},
           "features": FEATURES, "n_models": len(roster), "n_baselines": len(BASELINES),
           "null_95_upper": null_hi,
           "validation": {k: float(res_val[k]) for k in
                          ["MAE", "ProfitFactor", "RankIC", "RankIC_t", "DirAcc", "Skill"]},
           "test": {k: float(res_te[k]) for k in
                    ["MAE", "ProfitFactor", "RankIC", "RankIC_t", "DirAcc", "Skill"]}},
          open(f"{RESULTS}/model_meta.json", "w"), indent=1)

print("=" * 70)
print("MODEL SELECTION COMPLETE")
print("=" * 70)
print(f"  tried      : {len(roster)} models + {len(BASELINES)} baselines")
print(f"  selected   : {winner} ({all_results[winner]['family']})")
print(f"  validation : rank IC {res_val['RankIC']:+.4f} (t {res_val['RankIC_t']:+.2f})")
print(f"  test       : rank IC {res_te['RankIC']:+.4f} (t {res_te['RankIC_t']:+.2f})")
print(f"  chance     : 95% of random predictions fall below {null_hi:+.4f}")
print(f"\n  written to modelling/results/")